# 07 · The search

NSGA-II over the PyNAS search space (`[Space]` of the configuration), `[Search] population` = 50 candidates per generation, `[Search] generations` = 3. For every candidate, before any training:

- the weights in FP16 over 5 MB → infeasible, nothing compiled;
- ONNX → FP16 IR (OpenVINO 2022.3.1 Model Optimizer) → compiled for MYRIAD with `compile_tool`; not compiled, IR over 5 MB, or IR layer types missing from the ArgOS list → infeasible;
- predicted Myriad X latency (nn-Meter).

An infeasible candidate is never trained; another is drawn. The feasible ones are trained (10,000 batches of 16) in FP16 mixed precision with distillation from the teacher, BatchNorm folded into the convolutions for the last 20% of the steps, and scored with the folded network in FP16 on the search split. Selection on (fresh-burn IoU ↑, latency ↓). The winner: the best fitness of all trained candidates, ties broken by latency (`[Final] pick`), a rule fixed before the search runs.

The search runs in a background process of its own; this notebook can be closed while it runs, and every cell that reads results can run at any time.

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
for p in (os.path.join(os.getcwd(), "pyqnas", "src"), os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
import pynas
from pynas import config, report
from pynas.hardware import myriad
from pynas.search import driver

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False})
CONFIG = Path("configs/burnscar.ini")
cfg = config.load(CONFIG)
OUT = Path(cfg.get("Run", "out")) / "search"
soft = Path(cfg.get("Data", "pool")) / f"{cfg.get('Data', 'train')}_{cfg.get('Data', 'teacher')}.npy"
assert soft.exists(), f"no soft labels ({soft}): run 06_pool_and_teacher first"
st = myriad.status(cfg)
print(f"PyNAS {pynas.__version__}; Myriad tools: {st}; {torch.cuda.device_count()} GPUs")
if not all(st.values()):
    print("missing tools: myriad.setup(cfg) makes them (a few minutes, once per machine)")

## 1. Smoke test

The whole loop at a tiny size on this machine (4 candidates, 2 generations, 300 batches each), into `models/pynas/smoke/`: every stage runs on the real tools before the real search. About 10 minutes. Its results mean nothing and are not used.

In [ ]:
# [2] smoke test (in this notebook, about 10 minutes)
SMOKE = Path(cfg.get("Run", "out")) / "smoke"
cfg_smoke = config.with_overrides(cfg, {"Search.population": 4, "Search.generations": 2, "Train.steps": 300,
                                        "Train.log_every": 100})
w = driver.Search(cfg_smoke, SMOKE).run()
print(json.dumps(w, indent=1, default=str))
display(report.generation_summary(SMOKE))

## 2. The search

`launch` starts it in the background and returns at once; launching again a stopped search resumes it. About 15 hours on 2 GPUs.

In [ ]:
# [3] start the search
driver.launch(CONFIG, OUT)

In [ ]:
# [4] where it is (run again at any time)
info, runs = driver.status(OUT)
print(json.dumps(info, indent=1, default=str))
if runs is not None:
    display(runs.drop(columns=["model"]).tail(12))

## 3. Results

Per generation: chromosomes drawn, infeasible ones and why, candidates trained, best and median fitness. Then every trained candidate, fitness against predicted latency with the Pareto front, and the winner.

In [ ]:
# [5] the generations
report.generation_summary(OUT)

In [ ]:
# [6] the trained candidates, best first
t = report.search_table(OUT)
t[["gen", "score", "latency_ms", "ir_mb", "params", "minutes", "not_finite_steps", "spec"]].head(15)

In [ ]:
# [7] fitness against latency
fig, ax = plt.subplots(figsize=(8, 5))
report.plot_search(OUT, ax)
plt.tight_layout(); plt.show()

In [ ]:
# [8] the winner
w = json.load(open(OUT / "winner.json")) if (OUT / "winner.json").exists() else {"winner": "not yet"}
print(json.dumps(w, indent=1, default=str))

Next: `08_final_and_test.ipynb`, the winner trained again with 3 seeds, baselines, ablations, export and test.